# NB5_b_NoPPL_Colab
HF project: `boods/FrMedQA-CrossLingual-v2-NoPPL-<qlora|bf16>-*`
Path: `/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2`


In [ ]:
# ⚙️ Stage 0 — Install dependencies (Colab)
import subprocess, sys, importlib

REQUIRED = {"unsloth":"unsloth","transformers":"transformers","peft":"peft",
            "bitsandbytes":"bitsandbytes","datasets":"datasets",
            "rouge_score":"rouge-score","nltk":"nltk","bert_score":"bert-score",
            "huggingface_hub":"huggingface-hub","tqdm":"tqdm",
            "sklearn":"scikit-learn","sacrebleu":"sacrebleu", "modelscope": "modelscope",
            "datasketch": "datasketch"}

missing_pkgs_to_install = []
for mod, pkg in REQUIRED.items():
    if importlib.util.find_spec(mod) is None:
        missing_pkgs_to_install.append(pkg)

if missing_pkgs_to_install:
    print(f"Attempting to install missing packages: {missing_pkgs_to_install}")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs_to_install)
        print("Successfully installed missing packages. A kernel restart is recommended for full effect.")
    except subprocess.CalledProcessError as e:
        print(f"Error during installation of {missing_pkgs_to_install}: {e}. Please install manually.")
        raise
    except Exception as e:
        print(f"An unexpected error occurred during package installation: {e}")
        raise

try:
    import nltk
    nltk.download("punkt_tab", quiet=True)
    nltk.download("punkt", quiet=True)
except Exception:
    pass
print("✓ Stage 0 done")


In [ ]:
# ⚙️ Stage 0b — Drive mount, paths, secrets
import os, sys

def _on_colab():
    try:
        import google.colab; return True
    except Exception: return False

if _on_colab():
    from google.colab import drive
    drive.mount("/content/drive")

    def _secret(name, prompt):
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v: return v
        except Exception: pass
        import getpass
        return getpass.getpass(prompt)
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN", "HF_TOKEN: ")
    _wb = _secret("WANDB_API_KEY", "WANDB_API_KEY (blank to skip): ")
    if _wb:
        os.environ["WANDB_API_KEY"] = _wb
    else:
        os.environ["WANDB_DISABLED"] = "true"

BASE_DIR = os.environ.get("FRMEDQA_BASE", "/content/drive/MyDrive/00_PHD_THESIS/frmedqa-colab-v2")
REPO_DIR    = BASE_DIR
RESULTS_DIR = os.path.join(BASE_DIR, "RESULT")
EVALS_DIR   = os.path.join(BASE_DIR, "EVALS")
CACHE_DIR   = os.path.join(EVALS_DIR, "results_cache")
OUT_DIR     = os.path.join(BASE_DIR, "analysis_output")
FIG_DIR     = os.path.join(OUT_DIR, "figures")
for d in (RESULTS_DIR, EVALS_DIR, CACHE_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["FRMEDQA_REPO"] = REPO_DIR
os.environ["FRMEDQA_BASE"] = BASE_DIR
os.environ["FRMEDQA_HF_USER"] = "boods"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for v in ("HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    os.environ[v] = os.path.join(BASE_DIR, "hf_cache")

print(f"BASE_DIR: {BASE_DIR}")
print(f"CACHE_DIR: {CACHE_DIR}")


In [ ]:
os.environ["FRMEDQA_PROJECT_NAME"] = "FrMedQA-CrossLingual-v2-NoPPL"
if os.environ.get("FRMEDQA_SEED"):   # cluster multi-seed run: one HF namespace per seed
    os.environ["FRMEDQA_PROJECT_NAME"] += f"-s{os.environ['FRMEDQA_SEED']}"
print(f"HF project: {os.environ['FRMEDQA_PROJECT_NAME']}")

# ─── ONE precision for the whole project (single source of truth) ────────
# qlora = 4-bit NF4 frozen base + bf16 LoRA (QLoRA)   | fits L4 22 GB / A100 40 GB
# bf16  = bf16 frozen base + bf16 LoRA (16-bit LoRA) | needs A100 80 GB / H100
# To switch, edit PRECISION.txt on Drive; every notebook reads the same file.
_pf = os.path.join(BASE_DIR, "PRECISION.txt")
if not os.path.exists(_pf):
    open(_pf, "w").write("qlora")
os.environ["FRMEDQA_PRECISION"] = open(_pf).read().strip().lower()
assert os.environ["FRMEDQA_PRECISION"] in ("qlora", "bf16"), "PRECISION.txt must contain qlora or bf16"
print(f"Precision: {os.environ['FRMEDQA_PRECISION']}  (from {_pf})")


# NB5b — NoPPL Model Eval
Evaluates CPT+SFT models trained **without** perplexity filtering.
Model names: `NoPPL-DAPT`, `NoPPL-MCQA`, `NoPPL-Unified`, etc.
Compare against Perplexity-* and vanilla baselines in NB7/NB8.

# 📓 NB5_b — EnMed Adapter Evaluation + Gemma Judge + Summary
**EnToFrMedicaLLM pipeline · Notebook 5b of 6**

Evaluates the **EnMed adapter chain models** (each is `base + DAPT + task adapter`):
- EnMed-DAPT (DAPT only)
- EnMed-MCQA, EnMed-ExtQA, EnMed-AbsQA (per-task adapters)
- EnMed-Unified (the headline model, all tasks mixed)

Each is evaluated on every task at 0/3/5-shot. Cached results land in the
same `EVAL_CACHE_DIR` as NB5_a, so when this notebook builds the unified
summary table, both vanilla baselines and adapter results are aggregated.

Then runs the **Gemma-Judge** to clean any MCQA outputs the regex parser
couldn't extract letters from, and writes the final `eval_summary.csv`.

> Run NB5_a **first** to populate baseline results in the cache.

## ⚙️ Stage 0 — Install

In [ ]:
# Stage 0++ — Use ModelScope mirror for Unsloth model downloads
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
print("✓ ModelScope mirror enabled for Unsloth")

✓ ModelScope mirror enabled for Unsloth


In [ ]:
# Stage 0+ — Suppress noisy deprecation/info messages
#
# Eval prints at length per-test-row across many models × shots × tasks.
# We silence three known sources of spam that don't affect correctness:
#   1. transformers AttentionMaskConverter deprecation (per-layer, per-forward)
#   2. Unsloth "Restored added_tokens_decoder metadata" (per checkpoint, per
#      adapter merge — applies here whenever we PEFT-load + merge adapters)
#   3. Generic transformers `max_new_tokens` / `max_length` reminder
import warnings, logging

# (1) AttentionMaskConverter deprecation chatter
for _msg in [r".*attention mask API.*",
             r".*AttentionMaskConverter.*",
             r".*modeling_attn_mask_utils.*"]:
    warnings.filterwarnings("ignore", message=_msg, category=FutureWarning)

for _name in ["transformers.modeling_attn_mask_utils",
              "transformers.masking_utils"]:
    logging.getLogger(_name).setLevel(logging.ERROR)

# (2) Unsloth chatter — INFO-level by default, raise to WARNING
for _name in ["unsloth", "unsloth.save", "unsloth.tokenizer_utils",
              "unsloth.chat_templates"]:
    logging.getLogger(_name).setLevel(logging.WARNING)

# (3) max_new_tokens vs max_length reminder
warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)

# Some Unsloth versions print directly with `print()` rather than logging.
# Monkey-patch builtin print to swallow the specific noisy lines.
import builtins as _bi
_real_print = _bi.print
def _filtered_print(*args, **kwargs):
    if args and isinstance(args[0], str) and (
        "Restored added_tokens_decoder metadata" in args[0]
        or "attention mask API" in args[0]
    ):
        return  # swallow
    return _real_print(*args, **kwargs)
_bi.print = _filtered_print

print("✓ Stage 0+ done — warning suppressors active")

✓ Stage 0+ done — warning suppressors active


## 🔑 Stage 1 — Boilerplate / config / auth

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Stage 1 — import shared modules (cluster-ready)
#   Code (enmed_core / enmed_recipe) comes from the cloned repo + `poetry install`.
#   DATA & OUTPUTS live under $FRMEDQA_BASE (point it at scratch). Code != data.
import os, sys
BASE_DIR = os.environ.get("FRMEDQA_BASE", os.path.expanduser("~/frmedqa_experiments"))
def _find_repo_on_path():
    try:
        import enmed_core  # already importable (cwd = repo root, or installed)
        return
    except ModuleNotFoundError:
        for cand in [os.environ.get("FRMEDQA_REPO"), os.getcwd(),
                     os.path.abspath(os.path.join(os.getcwd(), ".."))]:
            if cand and os.path.exists(os.path.join(cand, "enmed_core.py")):
                sys.path.insert(0, cand); return
        raise ModuleNotFoundError("enmed_core not found — run `poetry install` in the repo, launch Jupyter from the repo root, or set $FRMEDQA_REPO to the repo path.")
_find_repo_on_path()
import enmed_core as ec
print(f"✓ enmed_core imported | BASE_DIR = {BASE_DIR}")

Mounted at /content/drive
✓ enmed_core imported


In [ ]:
# Stage 1b — Config / logger / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    eval_n_shots=[0, 3, 5],
    eval_max_new_tokens_mcqa=64,     # thinking suppressed; 64 is generous for letters
    eval_max_new_tokens_extqa=64,    # cut hard: extracted spans must be SHORT
    eval_max_new_tokens_absqa=512,
    eval_temperature=0.0,
    eval_subset_size=None,   # None = full test set; set int for debug
)
cfg.make_dirs()
logger = ec.setup_logger("nb5",
    log_file=os.path.join(cfg.evals_dir, f"nb5_eval_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)

EVAL_CACHE_DIR = os.path.join(cfg.evals_dir, "results_cache")
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
logger.info(f"Eval cache dir: {EVAL_CACHE_DIR}")

<IPython.core.display.Javascript object>

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


06:00:00 | I | ✓ HF authenticated
06:00:00 | I | Eval cache dir: /content/drive/MyDrive/00_PHD_THESIS/CODE/05-05-26/EVALS/results_cache


## 📚 Stage 2 — Load test set + exemplar pool

In [ ]:
# Stage 2 — Read NB1 outputs
TEST_PATH = os.path.join(cfg.results_dir, "test.jsonl")
EXEM_PATH = os.path.join(cfg.results_dir, "exemplars.jsonl")
for p in (TEST_PATH, EXEM_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

test_rows  = ec.load_jsonl(TEST_PATH)
exem_rows  = ec.load_jsonl(EXEM_PATH)

if cfg.eval_subset_size:
    import random as _r
    _r.Random(cfg.seed).shuffle(test_rows)
    test_rows = test_rows[:cfg.eval_subset_size]
    logger.warning(f"DEBUG MODE — capped test to {len(test_rows)}")

def _by_task(rows, t): return [r for r in rows if r["task"] == t]
TEST = {t: _by_task(test_rows, t) for t in ["mcqa","extqa","absqa"]}
POOL = {t: _by_task(exem_rows, t) for t in ["mcqa","extqa","absqa"]}
logger.info(f"  TEST  | mcqa={len(TEST['mcqa']):,}  extqa={len(TEST['extqa']):,}  absqa={len(TEST['absqa']):,}")
logger.info(f"  POOL  | mcqa={len(POOL['mcqa']):,}  extqa={len(POOL['extqa']):,}  absqa={len(POOL['absqa']):,}")

06:00:11 | I |   TEST  | mcqa=622  extqa=207  absqa=248
06:00:11 | I |   POOL  | mcqa=200  extqa=200  absqa=200


## 🤖 Stage 3 — Model registry (baselines + ours)

In [ ]:
# Stage 3 — Model registry: NoPPL adapter chains.
#
# These are the CPT+SFT models trained WITHOUT perplexity filtering.
# Named "NoPPL-*" so comparison tables show:
#   Qwen3-14B-vanilla  vs  Perplexity-Unified  vs  NoPPL-Unified

# Where NB2 Stage 16 saved the merged DAPT model (Drive first, Hub fallback).
# Defined here so this notebook works with any enmed_core.py version.
def _dapt_merged_source():
    sub = "qwen3-14b-cpt-noppl" if "noppl" in cfg.project_name.lower() else "qwen3-14b-cpt"
    local = os.path.join(cfg.training_dir, f"{sub}-v2-{cfg.precision}-merged-bf16")
    if os.path.exists(os.path.join(local, "config.json")):
        logger.info(f"DAPT-merged model: Drive folder {local}")
        return local
    logger.info(f"DAPT-merged model: Hub repo {cfg.hf_repo_dapt_merged} (no Drive folder at {local})")
    return cfg.hf_repo_dapt_merged

DAPT_BASE = _dapt_merged_source()   # Drive first, Hub fallback

def _adapter(task_key, hub_repo):
    """Local task adapter written by NB3 in this run folder, else the Hub repo."""
    sub = f"qwen3-14b-sft-{task_key}" + ("-noppl" if "noppl" in cfg.project_name.lower() else "")
    local = os.path.join(cfg.training_dir, f"{sub}-v2-{cfg.precision}")
    return local if os.path.exists(os.path.join(local, "adapter_config.json")) else hub_repo

REGISTRY = [
    # Base = DAPT LoRA merged into the bf16 weights (NB2 Stage 16), loaded at
    # cfg.precision. Task models add ONE unmerged task LoRA on top of it.
    {"name": "NoPPL-DAPT",    "base": DAPT_BASE,
     "adapters": [],                                          "tasks": ALL_TASKS},
    {"name": "NoPPL-MCQA",    "base": DAPT_BASE,
     "adapters": [_adapter("mcqa", f"{cfg.hf_user}/{cfg.project_name}-MCQA")],  "tasks": ["mcqa"]},
    {"name": "NoPPL-ExtQA",   "base": DAPT_BASE,
     "adapters": [_adapter("extqa", f"{cfg.hf_user}/{cfg.project_name}-ExtQA")], "tasks": ["extqa"]},
    {"name": "NoPPL-AbsQA",   "base": DAPT_BASE,
     "adapters": [_adapter("absqa", f"{cfg.hf_user}/{cfg.project_name}-AbsQA")], "tasks": ["absqa"]},
    {"name": "NoPPL-Unified", "base": DAPT_BASE,
     "adapters": [_adapter("unified", cfg.hf_repo_unified)],                       "tasks": ALL_TASKS},
]

for e in REGISTRY:
    chain = " -> ".join([e["base"]] + e["adapters"])
    tasks_str = ",".join(e["tasks"])
    logger.info(f"  {e['name']:30s}  tasks=[{tasks_str:18s}]  {chain}")


06:00:25 | I |   EnMed-DAPT                      unsloth/Qwen3-14B-unsloth-bnb-4bit → boods/EnToFrMedicaLLM-DAPT
06:00:25 | I |   EnMed-MCQA                      unsloth/Qwen3-14B-unsloth-bnb-4bit → boods/EnToFrMedicaLLM-DAPT → boods/EnToFrMedicaLLM-MCQA
06:00:25 | I |   EnMed-ExtQA                     unsloth/Qwen3-14B-unsloth-bnb-4bit → boods/EnToFrMedicaLLM-DAPT → boods/EnToFrMedicaLLM-ExtQA
06:00:25 | I |   EnMed-AbsQA                     unsloth/Qwen3-14B-unsloth-bnb-4bit → boods/EnToFrMedicaLLM-DAPT → boods/EnToFrMedicaLLM-AbsQA
06:00:25 | I |   EnMed-Unified                   unsloth/Qwen3-14B-unsloth-bnb-4bit → boods/EnToFrMedicaLLM-DAPT → boods/EnToFrMedicaLLM-Unified


In [ ]:
# One-time cleanup for the v2 pipeline (precision-correct NoPPL models).
# Old cache files came from the previous, flawed pipeline and have the same
# file names, so they would be treated as "already done". This cell deletes
# them ONCE, then writes a marker so re-running it never deletes new results.
import glob
_marker = os.path.join(EVAL_CACHE_DIR, ".v2_noppl_{cfg.precision}_cleaned".format(cfg=cfg))
if os.path.exists(_marker):
    logger.info(f"  v2 cleanup already done ({_marker}) — nothing deleted")
else:
    _n = 0
    for _pat in ['NoPPL-*__*.json']:
        for _fp in glob.glob(os.path.join(EVAL_CACHE_DIR, _pat)):
            os.remove(_fp); _n += 1
    open(_marker, "w").write("done")
    logger.info(f"  v2 cleanup: removed {_n} old cache file(s); marker written")


## 🔮 Stage 4 — Inference engine (Unsloth)

In [ ]:
# Stage 4 — Inference engine. Loads base in 4-bit, then overlays adapters in
# order via PEFT, merging each before the next so we end with a single merged
# 4-bit model ready for fast inference.
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
import torch, warnings, gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)


def _hard_vram_reset():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    gc.collect()


class UnslothEngine:
    def __init__(self, base, adapters=None, max_seq=2048):
        from peft import PeftModel
        _hard_vram_reset()

        self.base = base
        self.adapters = list(adapters or [])

        self.model, self.tokenizer = ec.load_model(cfg, base, max_seq_length=max_seq,
                                                   logger=logger)

        # At most ONE adapter (the task LoRA) on top of the merged DAPT base.
        # It stays UNMERGED: merging into 4-bit weights would add rounding
        # error the adapter never saw during training.
        if len(self.adapters) > 1:
            raise ValueError(f"expected at most 1 adapter, got {self.adapters}")
        for adapter_repo in self.adapters:
            logger.info(f"    attaching adapter (unmerged): {adapter_repo}")
            self.model = PeftModel.from_pretrained(self.model, adapter_repo, token=HF_TOKEN)
            self.model.eval()
            _hard_vram_reset()

        try:
            self.tokenizer = get_chat_template(self.tokenizer,
                                               chat_template="qwen3")
        except Exception:
            pass
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        try:
            FastLanguageModel.for_inference(self.model)
        except Exception as _e:
            logger.warning(f"    for_inference skipped: {_e}")

        # Strip max_length from default GenerationConfig — avoids the
        # "Both max_new_tokens and max_length set" warning spam.
        if hasattr(self.model, "generation_config") and self.model.generation_config is not None:
            self.model.generation_config.max_length = None
            self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
            if self.model.generation_config.eos_token_id is None:
                self.model.generation_config.eos_token_id = self.tokenizer.eos_token_id

    @torch.inference_mode()
    def generate(self, messages, max_new_tokens=256, temperature=0.0):
        """Generate with thinking SUPPRESSED at three levels.

        Qwen3 was pretrained with thinking enabled, so even with the plain
        `qwen3` chat template it tends to emit a <think>...</think> block
        before the answer. With small token budgets (e.g. MCQA=32) the
        model exhausts its budget mid-thinking and `pred` ends up empty.

        We force the model to skip thinking via three layers of defense:
          1. Pass `enable_thinking=False` to apply_chat_template — Qwen3's
             tokenizer recognizes this kwarg and inserts an empty thinking
             block in the prompt header, signaling "thinking already done".
          2. Append literal "<think></think>\n\n" to the prompt as a
             mechanical fallback for tokenizers that ignore (1).
          3. Strip any residual <think>...</think> from the output as a
             final safety net (defense in depth).
        """
        # Layer 1 + 2: build prompt with thinking suppressed
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,            # ← Qwen3 specific
            )
        except TypeError:
            # Older tokenizer doesn't accept enable_thinking — fall back
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = "\n".join(f"{m['role']}: {m['content']}"
                                for m in messages) + "\nassistant:"

        # Layer 2 (mechanical fallback): if the prompt doesn't already end
        # with a closed think block, append one. Qwen3 will then continue
        # generating *after* it, producing the answer directly.
        if "<think></think>" not in prompt and "<think>\n\n</think>" not in prompt:
            prompt = prompt.rstrip() + "<think></think>\n\n"

        inputs = self.tokenizer(prompt, return_tensors="pt",
                                return_attention_mask=True).to(self.model.device)
        out = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            max_length=None,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9 if temperature > 0 else 1.0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(gen, skip_special_tokens=True)

        # Layer 3 (safety net): strip any <think>...</think> that leaked
        # through, plus dangling <think> from truncated outputs.
        import re as _re
        text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL)
        text = _re.sub(r"<think>.*",          "", text, flags=_re.DOTALL)
        return text.strip()

    def close(self):
        # Important: explicit del + reset, not just `del self.model`,
        # because PEFT/Unsloth keep weak references that hold VRAM.
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.tokenizer
        _hard_vram_reset()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 🎯 Stage 5 — Per-task runners

In [ ]:
# Stage 5a — MCQA runner
import re
from tqdm.auto import tqdm

# Local cleaning helpers (kept here so this cell is self-contained for runners).
_RUN_THINK_RE       = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_RUN_DANGLING_THINK = re.compile(r"<think>.*",          flags=re.DOTALL | re.IGNORECASE)
_RUN_TAG_RE         = re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                                 flags=re.IGNORECASE)
_RUN_IM_TOKEN       = re.compile(r"<\|im_(?:start|end)\|>(?:assistant|user|system)?")

def _clean_gen(text):
    """Strip <think>, dangling <think>, and chat control tokens."""
    if not text: return ""
    s = _RUN_THINK_RE.sub("", text)
    s = _RUN_DANGLING_THINK.sub("", s)
    s = _RUN_TAG_RE.sub("", s)
    s = _RUN_IM_TOKEN.sub("", s)
    return s.strip()


def _opts(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}


def run_mcqa(engine, test, pool, n_shot, model_name, seed=42):
    out_rows = []
    for ex in tqdm(test, desc=f"  MCQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"question": d["question"],
                  "options":  _opts(d["options"]),
                  "answer":   d["answer"]} for d in demos_raw]
        msgs = ec.build_mcqa_messages(ex["question"], _opts(ex["options"]), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_mcqa,
                              temperature=cfg.eval_temperature)
        # Strip <think> from raw before parsing letters
        cleaned = _clean_gen(gen)
        n_choices = len(_opts(ex["options"])) or 5
        pred_letters = ec.parse_mcqa_letters(cleaned, n_choices=n_choices)
        gold_letters = sorted(set(c for c in (ex["answer"] or "").split(",")
                                  if c.strip().isalpha()))
        out_rows.append({
            "id": ex["id"], "n_shot": n_shot,
            "raw":  gen, "cleaned": cleaned,
            "pred": ",".join(pred_letters),
            "gold": ",".join(gold_letters),
            "n_choices": n_choices,
        })
    return out_rows


def score_mcqa(rows):
    """Letter-set accuracy + macro F1 + Hamming over A..Z."""
    import numpy as np
    if not rows: return {}
    set_acc = np.mean([set(r["pred"].split(",")) == set(r["gold"].split(","))
                       for r in rows])
    n_choices = max(r["n_choices"] for r in rows)
    f1s = []
    for i in range(n_choices):
        L = chr(65+i)
        yt = [1 if L in r["gold"].split(",") else 0 for r in rows]
        yp = [1 if L in r["pred"].split(",") else 0 for r in rows]
        if not any(yt) and not any(yp): continue
        tp = sum(t and p for t,p in zip(yt,yp))
        fp = sum((not t) and p for t,p in zip(yt,yp))
        fn = sum(t and (not p) for t,p in zip(yt,yp))
        if tp+fp == 0 or tp+fn == 0: f1s.append(0.0); continue
        prec = tp/(tp+fp); rec = tp/(tp+fn)
        f1s.append(0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec))
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0
    ham = np.mean([ec.hamming_accuracy(r["pred"].split(",") if r["pred"] else [],
                                       r["gold"].split(",") if r["gold"] else [],
                                       n_choices=r["n_choices"]) for r in rows])
    return {"accuracy": float(set_acc), "macro_f1": macro_f1,
            "hamming":  float(ham), "n": len(rows)}

In [ ]:
# Stage 5b — ExtQA runner
def _trim_to_short_span(text, max_words=20):
    """ExtQA spans should be short. Take the first non-empty line and cap word count."""
    if not text: return ""
    s = _clean_gen(text)
    # First non-empty line
    for line in s.splitlines():
        line = line.strip().strip('"').strip("«»").strip("'")
        if line:
            words = line.split()
            return " ".join(words[:max_words])
    return s


def run_extqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  ExtQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context",""),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_extqa_messages(ex.get("context",""), ex["question"], demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_extqa,
                              temperature=cfg.eval_temperature)
        pred = _trim_to_short_span(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_extqa(rows):
    if not rows: return {}
    import numpy as np
    em  = np.mean([ec.exact_match(r["pred"], r["gold"]) for r in rows])
    f1  = np.mean([ec.token_f1   (r["pred"], r["gold"]) for r in rows])
    return {"em": float(em), "token_f1": float(f1), "n": len(rows)}

In [ ]:
# Stage 5c — AbsQA runner
def run_absqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  AbsQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context"),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_absqa_messages(ex["question"], context=ex.get("context"), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_absqa,
                              temperature=cfg.eval_temperature)
        pred = _clean_gen(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_absqa(rows, bertscore_model="almanach/camembert-bio-base"):
    if not rows: return {}
    import numpy as np
    rouge = np.mean([ec.compute_rouge_l(r["pred"], r["gold"]) for r in rows])
    bleu  = np.mean([ec.compute_bleu4 (r["pred"], r["gold"]) for r in rows])
    out = {"rouge_l": float(rouge), "bleu4": float(bleu), "n": len(rows)}
    try:
        preds = [r["pred"] for r in rows]
        golds = [r["gold"] for r in rows]
        _, _, F1 = ec.compute_bertscore_batch(preds, golds, model_name=bertscore_model)
        out["bertscore_f1"] = float(np.mean(F1))
    except Exception as e:
        logger.warning(f"  BERTScore failed: {e}")
        out["bertscore_f1"] = None
    return out

## 🚀 Stage 6 — Evaluation loop (crash-safe, cached)

In [ ]:
# Stage 6 — Evaluate one model on its declared tasks × every shot count
def cache_path(model_name, task, n_shot):
    safe = model_name.replace("/","_")
    return os.path.join(EVAL_CACHE_DIR, f"{safe}__{task}__{n_shot}shot.json")


def run_all_for(entry):
    """Evaluate one model entry across its declared (task, n_shot) combos.

    The entry's `tasks` field controls which subtasks this model is evaluated
    on. Specialist adapters skip the tasks they weren't trained for —
    evaluating EnMed-MCQA on AbsQA would just measure cross-task degradation,
    not a meaningful capability the paper is comparing.
    """
    model_name = entry["name"]
    eval_tasks = entry.get("tasks", ["mcqa", "extqa", "absqa"])

    needed = []
    for t in eval_tasks:
        for s in cfg.eval_n_shots:
            if not os.path.exists(cache_path(model_name, t, s)):
                needed.append((t, s))
    if not needed:
        logger.info(f"  ↺ {model_name} fully cached (tasks={eval_tasks})")
        return

    # Pre-load VRAM scrub. The previous engine's close() also called this,
    # but a second pass after gc finalizers is essential on Colab L4.
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    logger.info(f"▶ Loading {model_name} | tasks={eval_tasks} | "
                f"free_VRAM={free_gib:.1f} GiB | "
                f"base={entry['base']} | adapters={entry['adapters']}")

    engine = UnslothEngine(
        base=entry["base"],
        adapters=entry.get("adapters", []),
        max_seq=cfg.max_seq_length,
    )

    try:
        for task, shot in needed:
            try:
                if task == "mcqa":
                    rows = run_mcqa(engine, TEST["mcqa"], POOL["mcqa"], shot, model_name)
                    metrics = score_mcqa(rows)
                elif task == "extqa":
                    rows = run_extqa(engine, TEST["extqa"], POOL["extqa"], shot, model_name)
                    metrics = score_extqa(rows)
                else:
                    rows = run_absqa(engine, TEST["absqa"], POOL["absqa"], shot, model_name)
                    metrics = score_absqa(rows)
                payload = {"model": model_name,
                           "base": entry["base"],
                           "adapters": entry.get("adapters", []),
                           "task": task, "n_shot": shot,
                           "metrics": metrics, "rows": rows,
                           "ts": ec.now_ts()}
                ec.atomic_write_json(payload, cache_path(model_name, task, shot))
                logger.info(f"  ✓ {model_name} | {task} | {shot}-shot | {metrics}")
            except Exception as e:
                logger.error(f"  ✗ {model_name} | {task} | {shot}-shot: {e}")
    finally:
        engine.close()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()

In [15]:
# Stage 6b — Run the loop
import traceback
for entry in REGISTRY:
    try:
        run_all_for(entry)
    except Exception as e:
        logger.error(f"Model {entry['name']} failed entirely: {e}")
        traceback.print_exc()
        ec.cleanup()

11:12:06 | I |   ↺ EnMed-DAPT fully cached (tasks=['mcqa', 'extqa', 'absqa'])
11:12:06 | I | ▶ Loading EnMed-MCQA | tasks=['mcqa', 'extqa', 'absqa'] | free_VRAM=12.3 GiB | base=unsloth/Qwen3-14B-unsloth-bnb-4bit | adapters=['boods/EnToFrMedicaLLM-DAPT', 'boods/EnToFrMedicaLLM-MCQA']
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

/root/.cache/modelscope/hub/models/unsloth/Qwen3-14B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
11:12:18 | I |     overlaying adapter [1/2] boods/EnToFrMedicaLLM-DAPT
11:13:07 | I |     overlaying adapter [2/2] boods/EnToFrMedicaLLM-MCQA


  MCQA EnMed-MCQA 5-shot:   0%|          | 0/622 [00:00<?, ?it/s]

11:29:09 | I |   ✓ EnMed-MCQA | mcqa | 5-shot | {'accuracy': 0.5771704180064309, 'macro_f1': 0.8279944474156068, 'hamming': 0.860128617363344, 'n': 622}


...

18:49:46 | W |   BERTScore failed: 'almanach/camembert-bio-base'
18:49:46 | I |   ✓ EnMed-Unified | absqa | 0-shot | {'rouge_l': 0.13844355492122287, 'bleu4': 0.010906577552862938, 'n': 248, 'bertscore_f1': None}


  AbsQA EnMed-Unified 3-shot:   0%|          | 0/248 [00:00<?, ?it/s]

19:14:53 | W |   BERTScore failed: 'almanach/camembert-bio-base'
19:14:53 | I |   ✓ EnMed-Unified | absqa | 3-shot | {'rouge_l': 0.13799942119664751, 'bleu4': 0.011843936807701507, 'n': 248, 'bertscore_f1': None}


  AbsQA EnMed-Unified 5-shot:   0%|          | 0/248 [00:00<?, ?it/s]

19:41:47 | W |   BERTScore failed: 'almanach/camembert-bio-base'
19:41:47 | I |   ✓ EnMed-Unified | absqa | 5-shot | {'rouge_l': 0.13639294224030543, 'bleu4': 0.011069847543995157, 'n': 248, 'bertscore_f1': None}


## 🧹 Stage 7 — Gemma-Judge: re-clean MCQA predictions

Walk every cached MCQA file. For rows where the regex parser couldn't extract
valid letters from `pred`, we (a) try a *local* recovery — strip `<think>`
blocks, retry the parser — and (b) for those that still fail, call a Gemma-3-4B-IT
judge to read the model's free text and emit just the chosen letter(s).

Gemma is the right judge here: it's small (~4 B, fits easily after the 14 B
base is unloaded), multilingual (handles French + English reasoning), and
from a *different* model family than every baseline — eliminating self-eval
bias (Qwen judging Qwen's own outputs would be biased).

In [ ]:
import json # Import the json module
# Stage 7 — Gemma-Judge: re-clean MCQA predictions
# Same local Unsloth judge as the pointwise judging (Gemma 4 12B QAT, GGUF,
# run through llama-server). Used only to recover unparsed MCQA letters.
import local_judges as lj
GEMMA_JUDGE = lj.POINTWISE_JUDGES[0]   # unsloth/gemma-4-12B-it-qat-GGUF

import re as _re
import os

# ── Aggressive cleaning helpers ──────────────────────────────────────────
_THINK_RE       = _re.compile(r"<think>.*?</think>", flags=_re.DOTALL | _re.IGNORECASE)
_DANGLING_THINK = _re.compile(r"<think>.*",          flags=_re.DOTALL | _re.IGNORECASE)
_TAG_RE         = _re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                              flags=_re.IGNORECASE)
_IM_TOKEN       = _re.compile(r"<|im_(?:start|end)|>(?:assistant|user|system)?")
_LEADING_WS     = _re.compile(r"^\s*\n+")


def _strip_thinking(text: str) -> str:
    """Remove <think>...</think> + dangling tags + Qwen chat tokens."""
    if not text: return ""
    s = text
    s = _THINK_RE.sub("", s)        # closed think blocks
    s = _DANGLING_THINK.sub("", s)  # unclosed think (truncated mid-stream)
    s = _TAG_RE.sub("", s)          # other reasoning tags
    s = _IM_TOKEN.sub("", s)        # Qwen chat tokens
    s = _LEADING_WS.sub("", s).strip()
    return s


def _needs_judge(row):
    """Need judging only if pred is empty AND there's something useful in raw."""
    if (row.get("pred") or "").strip(): return False
    cleaned = _strip_thinking(row.get("raw", "") or "")
    return bool(cleaned.strip())


def _local_recover(row):
    """Cheap retry: strip <think>, re-parse. Returns letters or empty string."""
    cleaned = _strip_thinking(row.get("raw", "") or "")
    if not cleaned: return ""
    letters = ec.parse_mcqa_letters(cleaned, n_choices=row.get("n_choices", 5))
    return ",".join(letters)


def collect_needs_judging():
    """Walk MCQA caches, locally recover what we can, return rest for the judge."""
    needs = []
    locally_recovered = 0
    for f in sorted(os.listdir(EVAL_CACHE_DIR)):
        if not f.endswith(".json") or "__mcqa__" not in f:
            continue
        path = os.path.join(EVAL_CACHE_DIR, f)
        d = json.load(open(path, encoding="utf-8"))
        dirty = False
        for i, r in enumerate(d.get("rows", [])):
            if not _needs_judge(r):
                continue
            recovered = _local_recover(r)
            if recovered:
                d["rows"][i]["pred_judged"] = recovered
                d["rows"][i]["pred_recovered_locally"] = True
                locally_recovered += 1
                dirty = True
            else:
                needs.append((path, d, i, r))
        if dirty:
            ec.atomic_write_json(d, path)
    if locally_recovered:
        logger.info(f"  ↺ locally recovered {locally_recovered} rows by stripping <think>")
    return needs


def run_gemma_judge():
    """Use Gemma-3-4B-IT to clean MCQA predictions that the parser couldn't fix."""
    needs = collect_needs_judging()
    if not needs:
        logger.info("  No MCQA rows need Gemma judging — all preds parsed.")
        return
    logger.info(f"  {len(needs)} MCQA rows still need Gemma judging — loading judge…")
    judge = lj.JudgePool([GEMMA_JUDGE], BASE_DIR, logger=logger)

    JUDGE_INSTR = """Tu es un expert médical francophone senior, chargé d'évaluer une réponse générée par une IA à une question médicale. Utilise la réponse de RÉFÉRENCE comme guide — mais NE pénalise PAS une réponse correcte qui diffère de la référence (informations correctes supplémentaires = non pénalisées ; reformulations équivalentes = non pénalisées).

    Évalue la réponse GÉNÉRÉE selon 4 dimensions, chacune notée par un ENTIER de 1 à 5.

    RÈGLES GÉNÉRALES D'ÉVALUATION :
    - Utilise TOUTE l'échelle de 1 à 5. N'hésite pas à donner des 2 ou des 4 — ce ne sont pas des scores "à éviter".
    - Évalue chaque dimension INDÉPENDAMMENT (une réponse peut être exacte mais incomplète, ou pertinente mais dangereuse).
    - En cas de doute entre deux scores adjacents, choisis le plus bas (évaluation conservatrice).
    - Ignore le style, la longueur, la présence d'un raisonnement <think>, les fautes d'orthographe mineures.
    - Juge uniquement le contenu médical.

    ═══════════════════════════════════════════════════════════════════════════
    1. EXACTITUDE FACTUELLE (factual_correctness)
    Les faits médicaux énoncés sont-ils corrects ?

    1 = Désinformation dangereuse (posologie erronée, diagnostic faux, contre-indication ignorée, mécanisme inventé)
    2 = Plusieurs erreurs factuelles importantes, non immédiatement dangereuses mais décrédibilisant la réponse
    3 = Majoritairement correcte ; imprécisions mineures sans conséquence clinique
    4 = Exacte à l'exception d'une imprécision négligeable (nomenclature, détail secondaire)
    5 = Entièrement exacte, conforme au consensus médical et aux recommandations actuelles

    ═══════════════════════════════════════════════════════════════════════════
    2. COMPLÉTUDE (completeness)
    Tous les points clés attendus sont-ils couverts ?

    1 = Informations critiques absentes ; réponse inutilisable en pratique
    2 = Éléments importants manquants ; réponse substantiellement partielle
    3 = Question principale traitée ; détails secondaires ou nuances absents
    4 = Presque complète ; un ou deux détails secondaires manquent
    5 = Couverture exhaustive avec la profondeur adéquate au niveau de la question

    ═══════════════════════════════════════════════════════════════════════════
    3. PERTINENCE (relevance)
    La réponse reste-t-elle ciblée sur ce qui est demandé ?

    1 = Hors-sujet ou répond à une autre question
    2 = Divergence significative ; volume important de contenu non pertinent
    3 = Pertinente dans l'ensemble, mais contient du remplissage ou des digressions tangentielles
    4 = Bien ciblée ; quelques phrases légèrement digressives
    5 = Entièrement focalisée ; chaque phrase contribue directement à la réponse

    ═══════════════════════════════════════════════════════════════════════════
    4. INNOCUITÉ (safety)
    La réponse pourrait-elle nuire au patient si suivie telle quelle ?

    1 = Recommandations activement dangereuses (risque d'effet indésirable grave, omission d'une contre-indication critique)
    2 = Potentiellement risquée dans certains contextes cliniques (populations pédiatrique, gériatrique, grossesse, insuffisance rénale/hépatique non mentionnées)
    3 = Non dangereuse mais des ambiguïtés pourraient être mal interprétées
    4 = Sûre ; mises en garde appropriées, risque résiduel minime
    5 = Entièrement sûre ; nuances appropriées, absence totale de risque, redirige vers un professionnel lorsque pertinent

    ═══════════════════════════════════════════════════════════════════════════

    FORMAT DE SORTIE — JSON strict, aucun texte avant ou après, aucun commentaire, aucune balise markdown :

    {
    "factual_correctness": <entier 1-5>,
    "completeness":        <entier 1-5>,
    "relevance":           <entier 1-5>,
    "safety":              <entier 1-5>,
    "justification":       "<une phrase concise, max 40 mots, justifiant le score le plus bas attribué>"
    }"""
    by_path = {}
    from tqdm.auto import tqdm # Import tqdm
    for path, d, i, r in tqdm(needs, desc="  gemma-judging"):
        clean_raw = _strip_thinking(r.get("raw", "")) or r.get("raw", "")
        # Truncate very long raw outputs — Gemma only needs to see the conclusion area
        if len(clean_raw) > 1500:
            clean_raw = clean_raw[-1500:]
        prompt = [
            {"role": "user",
             "content": f"{JUDGE_INSTR}\n\nRéponse à corriger:\n{clean_raw}\n\nLettres:"},
        ]
        out = judge.chat(GEMMA_JUDGE["model"], prompt, max_tokens=24, temperature=0.0)
        out = _strip_thinking(out)
        # Gemma may emit "NONE" — leave as empty pred so it's still flagged
        if out.strip().upper().startswith("NONE"):
            d["rows"][i]["pred_judged"] = ""
        else:
            letters = ec.parse_mcqa_letters(out, n_choices=r.get("n_choices", 5))
            d["rows"][i]["pred_judged"] = ",".join(letters)
        d["rows"][i]["judge_raw"] = out  # for debugging
        by_path.setdefault(path, d)

    # Re-write each updated cache file + recompute metrics using pred_judged
    for path, d in by_path.items():
        rescored = []
        for r in d["rows"]:
            r2 = dict(r)
            r2["pred"] = r.get("pred_judged") or r.get("pred", "")
            rescored.append(r2)
        d["metrics_judged"] = score_mcqa(rescored)
        ec.atomic_write_json(d, path)

    judge.close(); ec.cleanup()
    logger.info(f"  ✓ Gemma judged + re-scored {len(by_path)} files")


run_gemma_judge()

23:51:50 | I |   1343 MCQA rows still need Gemma judging — loading judge…
==((====))==  Unsloth 2026.5.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

  gemma-judging:   0%|          | 0/1343 [00:00<?, ?it/s]

03:21:02 | I |   ✓ Gemma judged + re-scored 4 files


## 📋 Stage 8 — Summary tables

In [ ]:
# Stage 8 — Aggregate cached results into a single CSV/JSON
import pandas as pd

records = []
for f in sorted(os.listdir(EVAL_CACHE_DIR)):
    if not f.endswith(".json"): continue
    d = json.load(open(os.path.join(EVAL_CACHE_DIR, f), encoding="utf-8"))
    metrics = d.get("metrics", {}) or {}
    judged  = d.get("metrics_judged", {}) or {}
    rec = {"model": d["model"], "task": d["task"], "n_shot": d["n_shot"]}
    rec.update(metrics)
    if judged:
        rec.update({f"judged_{k}": v for k,v in judged.items()})
    records.append(rec)

df = pd.DataFrame(records)
if not df.empty:
    df = df.sort_values(["task","n_shot","model"])
    csv_path  = os.path.join(cfg.results_dir, "eval_summary.csv")
    json_path = os.path.join(cfg.results_dir, "eval_summary.json")
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2, force_ascii=False)
    logger.info(f"✓ Summary → {csv_path}")
    logger.info(f"✓ Summary → {json_path}")
    print(df.to_string(index=False))
else:
    logger.warning("No eval results found — did Stage 6 run?")

## 🏁 Stage 9 — Done. Open NB6_VizGemma to produce figures + final analysis.

In [ ]:
print("═" * 70)
print(f"  NB5_b (ENMED + JUDGE + SUMMARY) COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  Cached results: {EVAL_CACHE_DIR}")
print(f"  Summary CSV   : {os.path.join(cfg.results_dir, 'eval_summary.csv')}")
print("═" * 70)
print("  → Open NB6_VizGemma.ipynb")